In [ ]:
import pandas as pd
import json

print("Loading the divided dataset...")
train_df = pd.read_json('/content/train_data.jsonl', lines=True)
test_df = pd.read_json('/content/test_data.jsonl', lines=True)

print("Generating digital indexes for users and products...")
# 1. Extract all individual users and products from the training set
unique_users = train_df['user_id'].unique()
unique_items = train_df['parent_asin'].unique()

# 2. Create a mapping dictionary (string -> starting from 0, numeric index)
user2id = {user: idx for idx, user in enumerate(unique_users)}
item2id = {item: idx for idx, item in enumerate(unique_items)}

print(f"A total of {len(user2id)} distinct users and {len(item2id)} distinct items were identified in the training set.")

# 3. Apply the mapping to the training set and the test set
train_df['user_idx'] = train_df['user_id'].map(user2id)
train_df['item_idx'] = train_df['parent_asin'].map(item2id)

test_df['user_idx'] = test_df['user_id'].map(user2id)
test_df['item_idx'] = test_df['parent_asin'].map(item2id)

# 4. Handling the cold-start data caused by the time series partitioning
# Because it is truncated by time, a very small number of products/users that first appeared in the test set will not be found in the dictionary of the training set. After mapping, they will become NaN.
missing_before = len(test_df)
test_df = test_df.dropna(subset=['user_idx', 'item_idx'])
test_df['user_idx'] = test_df['user_idx'].astype(int)
test_df['item_idx'] = test_df['item_idx'].astype(int)
drop_count = missing_before - len(test_df)

if drop_count > 0:
    print(f" In the test set, {drop_count} records were safely removed as they had not appeared in the training set (for error prevention purposes).")

# 5. Save the mapping dictionary and the processed data
print("Data and index dictionary are being saved for processing...")
with open('/content/user2id.json', 'w') as f:
    json.dump(user2id, f)
with open('/content/item2id.json', 'w') as f:
    json.dump(item2id, f)

train_df.to_json('/content/train_indexed.jsonl', orient='records', lines=True)
test_df.to_json('/content/test_indexed.jsonl', orient='records', lines=True)

print("Digital index conversion completed!")

Loading the divided dataset...
Generating digital indexes for users and products...
A total of 27503 distinct users and 16091 distinct items were identified in the training set.
 In the test set, 878 records were safely removed as they had not appeared in the training set (for error prevention purposes).
Data and index dictionary are being saved for processing...
Digital index conversion completed!
